<a id="inizio"></a>
# Esercitazione 1 · Soluzioni

⏱ ~60 min &nbsp;·&nbsp; Dati: nessuno

**In questo notebook impariamo a**

- costruire l'ambiente di un progetto con `uv add`, aggiungendo una libreria alla volta quando manca
- lanciare con `uv run` uno script che importa le sue funzioni da un altro file
- rimettere in forma il codice con `ruff format` e correggere gli avvisi di `ruff check`

<a id="indice"></a>
**Indice**

- [1. Domande](#sez-1)
- [2. Lo script delle temperature](#sez-2)
    - [Esercizio E1.1 · Il progetto e il primo lancio](#es-e1-1)
    - [Esercizio E1.2 · Aggiungere pandas](#es-e1-2)
    - [Esercizio E1.3 · Lo script completo](#es-e1-3)
    - [Esercizio E1.4 · Ruff trova i problemi](#es-e1-4)
    - [Esercizio E1.5 · Il codice in ordine](#es-e1-5)
    - [Esercizio E1.6 · La temperatura minima (facoltativo)](#es-e1-6)
    - [Esercizio E1.7 · Che cosa succede durante un import (facoltativo)](#es-e1-7)
    - [Esercizio E1.8 · L'ambiente ricostruito (facoltativo)](#es-e1-8)
- [3. Esercizi sui notebook 00-03, da fare a casa](#sez-3)
    - [3.1 Domande](#sez-3-1)
    - [3.2 Esercizi](#sez-3-2)
    - [Esercizio E1.9 · Massimo, minimo e differenza](#es-e1-9)
    - [Esercizio E1.10 · L'area del cerchio](#es-e1-10)
    - [Esercizio E1.11 · Lista invertita](#es-e1-11)
    - [Esercizio E1.12 · Posizioni pari](#es-e1-12)
    - [Esercizio E1.13 · Le capitali del G8](#es-e1-13)
    - [Esercizio E1.14 · Sfida: una riga sul G8 (facoltativo)](#es-e1-14)

In questa esercitazione rifai, passo per passo, la dimostrazione che abbiamo appena seguito sul conto della
spesa, lavorando nella cartella `Script_uv_Ruff/esercizio`. La cartella contiene un piccolo progetto già scritto,
con lo script `main.py` che usa le funzioni del file `meteo.py`, e il lavoro consiste nel costruirgli un
ambiente, una libreria alla volta, e nel mettere in ordine il codice con Ruff. Hai 45 minuti, poi guardiamo
insieme il risultato. La seconda parte del notebook raccoglie gli esercizi sui notebook 00-03 ed è
facoltativa, da fare a casa.

Questo notebook è la copia con le soluzioni. La cartella `Script_uv_Ruff/esercizio` che gli sta accanto,
dentro `Soluzioni/`, contiene il progetto com'è alla fine, con le librerie scritte in `pyproject.toml` e il
codice in ordine, e le celle degli esercizi riportano i valori che il terminale stampa. Per rifare
l'esercizio si lavora nella cartella `Modulo_1/Script_uv_Ruff/esercizio`.

Tutti i comandi si scrivono nel terminale di VS Code, aperto nella cartella `esercizio`. Il modo più
semplice per aprirlo è un clic con il tasto destro sulla cartella, nel pannello Explorer, e poi la voce
**Open in Integrated Terminal**; in alternativa, da un terminale aperto nella cartella del corso, si entra
nella cartella con `cd Modulo_1/Script_uv_Ruff/esercizio`. Gli script si lanciano sempre con `uv run`,
e non con il pulsante **Run Python File** di VS Code, che usa il Python del corso, dove pandas è già
installato, e quindi non farebbe comparire gli errori da cui parte l'esercizio.

<a id="sez-1"></a>
## 1. Domande

Le quattro domande che seguono riprendono i concetti del notebook *Ambiente, script e Ruff* che servono
negli esercizi sullo script, e richiedono pochi minuti. Per ciascuna conviene prevedere la risposta a mente e
solo dopo aprirla con un clic.

**Domanda 1.** In quale file sono elencate le librerie di cui ha bisogno un progetto gestito con uv?

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Risposta</summary>

In `pyproject.toml`, nella lista `dependencies` della sezione `[project]`. Il file `uv.lock`, che uv
scrive da solo, registra invece le versioni esatte di tutte le librerie del progetto, comprese quelle
da cui dipendono le librerie della lista.

</details>

**Domanda 2.** Con quale comando si aggiunge una libreria al progetto, e che cosa fa quel comando?

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Risposta</summary>

Con `uv add` seguito dal nome della libreria, per esempio `uv add pandas`. Il comando aggiunge la
libreria a `pyproject.toml`, aggiorna `uv.lock` e la installa nella cartella `.venv` del progetto.

</details>

**Domanda 3.** Che differenza c'è tra un formatter e un linter, e con quali comandi Ruff svolge i due compiti?

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Risposta</summary>

Il formatter rimette in forma il codice, sistemando spazi, righe vuote e a capo senza cambiare quello
che il programma fa, mentre il linter legge il codice senza eseguirlo e segnala gli errori probabili,
come un import inutilizzato o una variabile mai usata, e i problemi di stile che il formatter non
sistema da solo, come una riga troppo lunga. Ruff svolge entrambi i compiti: `ruff format` rimette in
forma il codice e `ruff check` lo controlla.

</details>

**Domanda 4.** Lo script `main.py` dell'esercizio contiene la riga `from meteo import media, massima`, anche se `meteo` non è una libreria installata. Perché la riga funziona?

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Risposta</summary>

Perché il file `meteo.py` sta nella stessa cartella di `main.py`, e Python cerca i moduli da importare
prima nella cartella dello script lanciato e solo dopo tra i moduli della libreria standard e le
librerie installate in `.venv`. Un file `.py` del progetto si importa quindi come una libreria, con il
nome del file senza l'estensione.

</details>

<a id="sez-2"></a>
## 2. Lo script delle temperature

La cartella `esercizio` è fatta come la cartella `demo` all'inizio della dimostrazione: contiene soltanto
due script, il file `meteo.py` con tre funzioni sulle temperature e lo script `main.py`, che le importa,
stampa la temperatura massima e quella media della settimana e salva una tabella in Excel. Il progetto,
con il suo `pyproject.toml` e il suo ambiente, lo creiamo noi. I cinque esercizi seguono l'ordine della
dimostrazione e vanno fatti in sequenza, perché ognuno parte dalla cartella come l'ha lasciata il
precedente.

Ogni esercizio comincia dicendo che cosa vogliamo ottenere e perché, e il blocco Passo per passo, che si
apre con un clic, elenca i comandi da lanciare; sotto c'è un suggerimento, nascosto anch'esso. Il lavoro si
fa nel terminale e nei file della cartella, mentre nella cella dell'esercizio si copia quello che il
terminale ha stampato. La verifica controlla quei valori e, negli esercizi che cambiano i file del
progetto, legge anche quei file, per confermare che i comandi abbiano avuto effetto. La cella `soluzione`
mostra i comandi e i valori attesi di quell'esercizio.

Prima di ogni comando conviene controllare che il terminale sia nella cartella `esercizio`, il cui
percorso compare all'inizio della riga di comando. Lanciato dalla cartella del corso, `uv add` lavorerebbe
sul progetto del corso, che ha già pandas e openpyxl, quindi non installerebbe nulla e il progetto
`esercizio` resterebbe senza librerie. Anche Ruff, lanciato da lì, metterebbe in ordine
i file di `esercizio` prima del tempo, e gli esercizi su Ruff non troverebbero più i quattro problemi da
cui partono.

Un avviso che nomina `VIRTUAL_ENV` indica soltanto che il terminale ha attivato l'ambiente del corso, e si
può ignorare, perché uv usa comunque l'ambiente della cartella in cui viene lanciato. Se vuoi
ricominciare l'esercizio da capo, cancella la cartella `esercizio` ed estraila di nuovo dallo zip del
corso, perché estraendola sopra quella vecchia resterebbero al loro posto `.venv`, `uv.lock` e gli altri
file creati dai comandi.

<a id="es-e1-1"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio E1.1 · Il progetto e il primo lancio**

Come nella dimostrazione, i due script diventano un progetto con `uv init --bare`, che crea nella
cartella soltanto il file `pyproject.toml` con il nome della cartella e la lista delle librerie vuota. Prima di crearlo
lanciamo lo script una volta, per vedere che senza un progetto suo uv usa l'ambiente del corso, dove
pandas c'è già. Dopo `uv init --bare` lo stesso comando crea l'ambiente del progetto e il file `uv.lock`, e lo
script si ferma subito con un `ModuleNotFoundError`, il cui messaggio nomina la libreria mancante.

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Passo per passo</summary>

1. Nel terminale aperto nella cartella `esercizio` lancia `uv run main.py`: lo script arriva in fondo,
   perché uv, non trovando un progetto nella cartella, risale fino a quello del corso.
2. Lancia `uv init --bare`. La prima riga, `Project ... is excluded by workspace`, dice che la cartella resta
   fuori dal progetto del corso, ed è quello che vogliamo. Apri il `pyproject.toml` appena creato e
   leggi il nome del progetto e la voce `dependencies`, che è una lista vuota.
3. Lancia di nuovo `uv run main.py`, poi guarda nel pannello Explorer la cartella `.venv` e il file
   `uv.lock`, comparsi accanto agli script.
4. Scrivi in `nome_progetto` il nome del progetto, in `cartella_creata` il nome della cartella che uv
   dice di aver creato e in `libreria_mancante` il nome della libreria che compare nell'ultima riga
   dell'errore.

</details>

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Suggerimento 1</summary>

Il nome del progetto è alla voce `name` di `pyproject.toml`. Tra le prime righe stampate dal secondo
`uv run` c'è `Creating virtual environment at: ...`, e l'ultima riga dell'errore ha la forma
`No module named '...'`.

</details>

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione E1.1**

</div>

In [ ]:
# nel terminale, nella cartella esercizio
#   uv run main.py        funziona, con l'ambiente del corso
#   uv init --bare        Initialized project `esercizio`
#   uv run main.py
# Creating virtual environment at: .venv
# ...
# ModuleNotFoundError: No module named 'pandas'
nome_progetto = "esercizio"
cartella_creata = ".venv"
libreria_mancante = "pandas"

In [ ]:
from corso import verifica
verifica("E1.1")

<a id="es-e1-2"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio E1.2 · Aggiungere pandas**

Lo script ha bisogno di pandas, e una libreria si aggiunge al progetto con `uv add`, che la scrive in
`pyproject.toml`, registra in `uv.lock` le versioni esatte e la installa in `.venv`. Vogliamo
aggiungerla, guardare che cosa è cambiato nei due file e lanciare di nuovo lo script. Questa volta lo
script stampa le prime due righe e si ferma più avanti, quando prova a salvare la tabella in Excel,
perché manca una seconda libreria.

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Passo per passo</summary>

1. Lancia `uv add pandas` e guarda l'elenco delle librerie installate, dove oltre a pandas compaiono
   quelle da cui pandas dipende, come numpy.
2. Apri `pyproject.toml` e controlla la voce `dependencies`. Poi apri `uv.lock`, che uv ha creato al
   primo `uv run` con il solo progetto, e cerca la riga `name = "pandas"`, che `uv add` ha aggiunto
   insieme a quelle delle librerie da cui pandas dipende.
3. Lancia di nuovo `uv run main.py` e scrivi in `seconda_libreria` il nome della libreria che manca
   adesso.

</details>

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Suggerimento 1</summary>

Il traceback è più lungo del primo, perché l'errore nasce dentro pandas, nel metodo `to_excel` che
scrive il file Excel. Il nome della libreria che manca si legge, come prima, nell'ultima riga.

</details>

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione E1.2**

</div>

In [ ]:
# nel terminale, nella cartella esercizio
#   uv add pandas        installa pandas e le librerie da cui dipende
#   uv run main.py
# Temperatura massima della settimana: 16.0 gradi
# Temperatura media della settimana: 15.0 gradi, calcolata su 5 giorni
# ...
# ModuleNotFoundError: No module named 'openpyxl'
seconda_libreria = "openpyxl"

In [ ]:
from corso import verifica
verifica("E1.2")

<a id="es-e1-3"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio E1.3 · Lo script completo**

Con la seconda libreria lo script arriva fino in fondo: stampa la temperatura massima e quella media
della settimana, salva la tabella nel file `meteo.xlsx` e lo conferma con una terza riga. Vogliamo
arrivare a questo punto e conservare nel notebook la riga della media, copiata dal terminale, come
prova che lo script è stato eseguito.

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Passo per passo</summary>

1. Lancia `uv add openpyxl` e poi di nuovo `uv run main.py`.
2. Controlla che nella cartella `esercizio` sia comparso il file `meteo.xlsx`.
3. Copia dal terminale la riga che comincia con `Temperatura media` e incollala tra le virgolette di
   `riga_media`.

</details>

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Suggerimento 1</summary>

Nel terminale di VS Code il testo si copia come in un editor, selezionandolo con il mouse e
premendo **Ctrl+C**.

</details>

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione E1.3**

</div>

In [ ]:
# nel terminale, nella cartella esercizio
#   uv add openpyxl
#   uv run main.py
# Temperatura massima della settimana: 16.0 gradi
# Temperatura media della settimana: 15.0 gradi, calcolata su 5 giorni
# La tabella delle temperature è nel file meteo.xlsx
riga_media = "Temperatura media della settimana: 15.0 gradi, calcolata su 5 giorni"

In [ ]:
from corso import verifica
verifica("E1.3")

<a id="es-e1-4"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio E1.4 · Ruff trova i problemi**

Lo script funziona, ma il codice di `meteo.py` e di `main.py` è scritto in modo disordinato e contiene
qualche riga che non serve, come un import inutilizzato e una variabile mai usata. Vogliamo aggiungere
Ruff al progetto come strumento di sviluppo, lasciare che il formatter sistemi la forma e poi leggere
che cosa segnala il linter. Dopo `ruff format` il comando `ruff check` trova quattro problemi, due in
ciascun file.

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Passo per passo</summary>

1. Lancia `uv add --dev ruff`, che mette Ruff nel gruppo `dev` di `pyproject.toml`, tra gli strumenti
   di chi scrive il codice, e non tra le librerie che servono allo script.
2. Lancia `uv run ruff format` e apri i due file per vedere che cosa è cambiato, per esempio gli spazi
   attorno agli operatori e le righe vuote tra le funzioni.
3. Lancia `uv run ruff check` e scrivi in `avvisi` i codici dei quattro problemi, come stringhe.

</details>

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Suggerimento 1</summary>

Ogni avviso si apre con il suo codice, una lettera seguita da tre cifre come `F401`, e sotto
riporta il file e la riga a cui si riferisce. La riga lunga che `ruff format` ha messo tra
parentesi resta segnalata, perché il formatter non divide le stringhe.

</details>

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione E1.4**

</div>

In [ ]:
# nel terminale, nella cartella esercizio
#   uv add --dev ruff
#   uv run ruff format       2 files reformatted
#   uv run ruff check
# I001 [*] Import block is un-sorted or un-formatted         --> main.py
# E501 Line too long (102 > 100)                             --> main.py
# F401 [*] `math` imported but unused                        --> meteo.py
# F841 Local variable `giorni` is assigned to but never used --> meteo.py
avvisi = ["I001", "E501", "F401", "F841"]

In [ ]:
from corso import verifica
verifica("E1.4")

<a id="es-e1-5"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio E1.5 · Il codice in ordine**

Dei quattro avvisi, due si correggono da soli con `ruff check --fix`, che toglie l'import inutilizzato
e riordina gli import, mentre gli altri due richiedono una decisione nostra. Ruff non cancella la
variabile mai usata, perché potrebbe essere il segno di un errore da capire, e non divide la stringa
troppo lunga. Vogliamo arrivare al punto in cui `uv run ruff check` risponde `All checks passed!` e lo
script stampa le stesse righe di prima.

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Passo per passo</summary>

1. Lancia `uv run ruff check --fix` e guarda come sono diventati gli import di `main.py`, con prima le
   librerie installate e poi i file del progetto, separati da una riga vuota.
2. In `meteo.py` cancella la riga `giorni = len(temperature)`.
3. In `main.py` dividi il testo del `print` della media in due f-string scritte una dopo l'altra dentro
   le parentesi, in modo che nessuna riga superi i 100 caratteri.
4. Lancia di nuovo `uv run ruff format`, che toglie la riga vuota rimasta al posto dell'import, e poi
   `uv run ruff check`.
5. Scrivi in `esito_finale` la riga stampata da `ruff check`, poi lancia `uv run main.py` e controlla
   che lo script stampi le stesse righe di prima.

</details>

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Suggerimento 1</summary>

Due stringhe scritte una dopo l'altra dentro le stesse parentesi diventano una stringa sola. Il
testo si può dividere dopo `gradi, `, lasciando lo spazio alla fine del primo pezzo, e ogni pezzo
che contiene delle graffe ha bisogno della sua `f`.

</details>

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione E1.5**

</div>

In [ ]:
# nel terminale, nella cartella esercizio
#   uv run ruff check --fix      corregge F401 e I001
# a mano, in meteo.py, si cancella la riga giorni = len(temperature)
# a mano, in main.py, il print della media diventa
#     print(
#         f"Temperatura media della settimana: {media(valori):.1f} gradi, "
#         f"calcolata su {len(valori)} giorni"
#     )
#   uv run ruff format
#   uv run ruff check            All checks passed!
#   uv run main.py
esito_finale = "All checks passed!"

In [ ]:
from corso import verifica
verifica("E1.5")

I tre esercizi che seguono sono facoltativi e lavorano sulla stessa cartella. Il primo aggiunge una riga
allo script, il secondo mostra che cosa fa Python quando importa un file e il terzo ricostruisce l'ambiente
da zero, per vedere che `pyproject.toml` e `uv.lock` bastano a ricrearlo identico.

<a id="es-e1-6"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio E1.6 · La temperatura minima (facoltativo)**

Il file `meteo.py` contiene anche la funzione `minima`, che `main.py` non usa. Vogliamo importarla e
aggiungere allo script una riga con la temperatura minima della settimana, scritta come le altre con
una cifra decimale: la riga stampata è `Temperatura minima della settimana: 13.8 gradi`.

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Passo per passo</summary>

1. Aggiungi `minima` ai nomi importati nella riga `from meteo import` di `main.py`.
2. Sotto il `print` della massima scrivi un `print` con una f-string che chiama `minima(valori)` con il
   formato `:.1f`.
3. Lancia `uv run main.py` e copia la nuova riga in `riga_minima`.
4. Lancia `uv run ruff check`; se segnala di nuovo `I001`, `uv run ruff check --fix` rimette in ordine
   alfabetico i nomi importati.

</details>

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Suggerimento 1</summary>

La riga nuova ha la stessa forma di quella della massima, con `minima` al posto di `massima`.

</details>

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione E1.6**

</div>

In [ ]:
# in main.py
#   from meteo import massima, media, minima
#   print(f"Temperatura minima della settimana: {minima(valori):.1f} gradi")
# nel terminale, nella cartella esercizio
#   uv run main.py
#   uv run ruff check            All checks passed!
riga_minima = "Temperatura minima della settimana: 13.8 gradi"

In [ ]:
from corso import verifica
verifica("E1.6")

<a id="es-e1-7"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio E1.7 · Che cosa succede durante un import (facoltativo)**

Quando `main.py` importa le funzioni di `meteo.py`, Python esegue tutto il file importato, dall'inizio
alla fine, prima di proseguire con le righe di `main.py`. La variabile speciale `__name__`, inoltre,
ha un valore diverso a seconda che il file venga importato o lanciato direttamente con
`uv run meteo.py`. Vogliamo vedere entrambe le cose con una riga di prova, perché spiegano come mai
negli script le istruzioni che fanno il lavoro del programma si mettono sotto
`if __name__ == "__main__":`.

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Passo per passo</summary>

1. In `meteo.py`, sotto la docstring, aggiungi la riga
   `print(f"Sto leggendo meteo.py, con il nome {__name__}")`.
2. Lancia `uv run main.py`, controlla che la nuova riga compaia prima di quelle di `main.py` e scrivi
   in `nome_importato` il nome che stampa.
3. Lancia `uv run meteo.py` e scrivi in `nome_lanciato` il nome che stampa questa volta.
4. Togli la riga di prova da `meteo.py`.

</details>

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione E1.7**

Quando `main.py` lo importa, `meteo.py` ha come nome `meteo`, cioè il nome del file senza
l'estensione, mentre lanciato con `uv run meteo.py` è il programma principale e Python lo chiama
`__main__`. Le righe scritte sotto `if __name__ == "__main__":` vengono quindi eseguite solo quando
il file è lanciato direttamente, e chi lo importa riceve le funzioni senza eseguire altro.

</div>

In [ ]:
# in meteo.py, sotto la docstring
#   print(f"Sto leggendo meteo.py, con il nome {__name__}")
# nel terminale, nella cartella esercizio
#   uv run main.py     la prima riga è: Sto leggendo meteo.py, con il nome meteo
#   uv run meteo.py    stampa: Sto leggendo meteo.py, con il nome __main__
nome_importato = "meteo"
nome_lanciato = "__main__"

In [ ]:
from corso import verifica
verifica("E1.7")

<a id="es-e1-8"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio E1.8 · L'ambiente ricostruito (facoltativo)**

Non serve conservare né condividere la cartella `.venv`, perché si ricrea in qualunque momento a
partire da `pyproject.toml` e `uv.lock`, con le stesse versioni di prima. Vogliamo provarlo
cancellando l'ambiente dell'esercizio e ricostruendolo con un solo comando, senza ripetere nessun
`uv add`.

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Passo per passo</summary>

1. Nel pannello Explorer di VS Code cancella la cartella `.venv` dentro `esercizio`, con il tasto destro
   e la voce **Delete**.
2. Lancia `uv sync`, che crea di nuovo `.venv` e installa le stesse librerie, Ruff compreso, di solito
   in pochi secondi, perché uv tiene una copia delle librerie già scaricate.
3. Lancia `uv run main.py` e controlla che lo script funzioni come prima.
4. Scrivi in `file_versioni` il nome del file da cui `uv sync` legge le versioni esatte da installare.

</details>

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione E1.8**

</div>

In [ ]:
# nel pannello Explorer: tasto destro su esercizio/.venv, Delete
# nel terminale, nella cartella esercizio
#   uv sync           Creating virtual environment at: .venv, poi installa le stesse librerie
#   uv run main.py
file_versioni = "uv.lock"

In [ ]:
from corso import verifica
verifica("E1.8")

<a id="sez-3"></a>
## 3. Esercizi sui notebook 00-03, da fare a casa

Questa seconda parte è facoltativa e si fa a casa, per ripassare i numeri, le stringhe, le liste e i
dizionari dei notebook 00-03. Comincia con quattro domande di riepilogo e prosegue con cinque esercizi
brevi, in difficoltà crescente, e con una sfida libera senza verifica. Gli esercizi funzionano come
quelli della prima parte, con il blocco Passo per passo e il suggerimento che si aprono con un clic, la
verifica e la cella `soluzione`, ma il codice si scrive direttamente nella cella del notebook.

<a id="sez-3-1"></a>
### 3.1 Domande

Per ciascuna delle quattro domande conviene prevedere la risposta a mente e solo dopo aprirla con un
clic. In caso di dubbio si può provare la riga in una cella di codice e, se il risultato sorprende,
rileggere la parte del notebook che tratta quell'argomento.

**Domanda 5.** Cosa restituisce `17 // 5`, e cosa restituisce `17 % 5`?

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Risposta</summary>

`17 // 5` restituisce `3`, perché la divisione intera tiene soltanto la parte intera del quoziente 3.4,
e `17 % 5` restituisce `2`, il resto della stessa divisione.

</details>

**Domanda 6.** Con `numeri = [3, 6, 9, 12, 15]`, cosa restituisce `numeri[1:3]`?

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Risposta</summary>

Restituisce `[6, 9]`. Lo slicing parte dall'indice 1 e si ferma prima dell'indice 3, quindi prende il
secondo e il terzo elemento.

</details>

**Domanda 7.** Con `punto = (45.46, 9.19)`, cosa succede se eseguiamo `punto[0] = 45.5`?

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Risposta</summary>

Python solleva un `TypeError`, perché le tuple sono immutabili e non permettono di sostituire un
elemento dopo la creazione. Per cambiare una coordinata si crea una tupla nuova.

</details>

**Domanda 8.** Con `capitali = {"Italia": "Roma", "Francia": "Parigi"}`, cosa restituisce `capitali["Italia"]` e cosa fa `capitali["Spagna"] = "Madrid"`?

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Risposta</summary>

`capitali["Italia"]` restituisce `"Roma"`, il valore associato a quella chiave. L'assegnazione a una
chiave che non esiste aggiunge la coppia al dizionario, che dopo ha tre elementi.

</details>

<a id="sez-3-2"></a>
### 3.2 Esercizi

I cinque esercizi ripercorrono i numeri, le stringhe e le strutture dati dei notebook 01 e 02. Ognuno lavora
sui dati scritti nella propria cella, quindi si può affrontare anche senza aver risolto i precedenti. La
sfida in fondo non ha una verifica, perché la forma della frase da stampare è libera.

<a id="es-e1-9"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio E1.9 · Massimo, minimo e differenza**

Nella cella sono definite tre variabili numeriche. Vogliamo trovare il valore massimo, il valore
minimo, la loro differenza e la media dei tre numeri arrotondata a un decimale, e stampare i quattro
risultati con una f-string ciascuno. Con i valori della cella il massimo è 25, il minimo 7, la
differenza 18 e la media 14.0.

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Passo per passo</summary>

1. Salva in `valore_massimo` e `valore_minimo` il più grande e il più piccolo dei tre numeri, con
   `max()` e `min()`, e in `differenza` la loro differenza.
2. Salva in `media` la media dei tre numeri, arrotondata a un decimale con `round()`.
3. Stampa i quattro risultati, uno per riga, con le f-string.

</details>

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Suggerimento 1</summary>

`max()` e `min()` accettano più argomenti separati da virgola; la media è la somma dei tre numeri
divisa per 3, e `round(valore, 1)` la arrotonda a un decimale.

</details>

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione E1.9**

</div>

In [ ]:
a = 10
b = 25
c = 7

valore_massimo = max(a, b, c)
valore_minimo = min(a, b, c)
differenza = valore_massimo - valore_minimo
media = round((a + b + c) / 3, 1)

print(f"Valore massimo: {valore_massimo}")   # Output: Valore massimo: 25
print(f"Valore minimo: {valore_minimo}")     # Output: Valore minimo: 7
print(f"Differenza: {differenza}")           # Output: Differenza: 18
print(f"Media: {media}")                     # Output: Media: 14.0

In [ ]:
from corso import verifica
verifica("E1.9")

<a id="es-e1-10"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio E1.10 · L'area del cerchio**

Vogliamo calcolare l'area e la circonferenza di un cerchio di raggio 5 e stampare l'area con due
decimali. L'area è π moltiplicato per il raggio al quadrato, la circonferenza è il doppio di π per il
raggio, e il valore di π lo fornisce il modulo `math`, con `math.pi`. La riga stampata è
`L'area del cerchio è: 78.54` e la circonferenza vale 31.41592653589793.

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Passo per passo</summary>

1. Salva in `area` il prodotto tra `math.pi` e il raggio al quadrato.
2. Stampa l'area con una f-string che la mostri con due decimali, usando il formato `{area:.2f}`.
3. Salva in `circonferenza` il doppio di π per il raggio.

</details>

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Suggerimento 1</summary>

Dentro le graffe di una f-string, dopo il nome della variabile, `:.2f` chiede due cifre
decimali; il valore di `area` non cambia, cambia soltanto come viene stampato.

</details>

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione E1.10**

</div>

In [ ]:
import math

raggio = 5
area = math.pi * raggio ** 2
circonferenza = 2 * math.pi * raggio

print(f"L'area del cerchio è: {area:.2f}")   # Output: L'area del cerchio è: 78.54
print(circonferenza)                          # Output: 31.41592653589793

In [ ]:
from corso import verifica
verifica("E1.10")

<a id="es-e1-11"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio E1.11 · Lista invertita**

Partendo dalla lista `[1, 2, 3, 4, 5]` vogliamo ricavare con lo slicing, senza riscrivere gli
elementi a mano, la lista letta al contrario, `[5, 4, 3, 2, 1]`, i suoi ultimi due elementi,
`[4, 5]`, e i suoi elementi di posizione pari, `[1, 3, 5]`.

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Passo per passo</summary>

1. Salva in `lista_invertita` la lista letta al contrario, con uno slicing a passo negativo.
2. Salva in `ultimi_due` gli ultimi due elementi della lista di partenza, con uno slicing che usa un
   indice negativo, e in `ogni_due` gli elementi di posizione pari.

</details>

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Suggerimento 1</summary>

Un indice negativo conta dalla fine, quindi `lista[-2:]` parte dal penultimo elemento; il passo
`2` prende un elemento ogni due.

</details>

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione E1.11**

</div>

In [ ]:
lista = [1, 2, 3, 4, 5]

lista_invertita = lista[::-1]
ultimi_due = lista[-2:]
ogni_due = lista[::2]

print(lista_invertita)  # Output: [5, 4, 3, 2, 1]
print(ultimi_due, ogni_due)  # Output: [4, 5] [1, 3, 5]

In [ ]:
from corso import verifica
verifica("E1.11")

<a id="es-e1-12"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio E1.12 · Posizioni pari**

La lista della cella alterna nomi di donne e di uomini, a partire da Elena. Vogliamo selezionare con
uno slicing tutte le donne tranne Elena, cioè gli elementi in posizione pari esclusa la posizione 0,
e con un secondo slicing tutti gli uomini, che stanno nelle posizioni dispari, per poi contarli. Le
donne selezionate sono `['Martina', 'Giulia', 'Francesca', 'Sara']`, e gli uomini sono cinque, da
`'Luigi'` a `'Lorenzo'`.

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Passo per passo</summary>

1. Salva in `elementi_pari` gli elementi in posizione pari escludendo la posizione 0.
2. Salva in `uomini` gli elementi in posizione dispari e in `quanti_uomini` quanti sono.

</details>

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Suggerimento 1</summary>

Le posizioni dispari si prendono partendo dall'indice 1 con passo 2, e `len()` conta gli
elementi di una lista.

</details>

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione E1.12**

</div>

In [ ]:
lista = ["Elena", "Luigi", "Martina", "Marco", "Giulia", "Simone", "Francesca", "Alessandro", "Sara", "Lorenzo"]

elementi_pari = lista[2::2]
uomini = lista[1::2]
quanti_uomini = len(uomini)

print(elementi_pari)  # Output: ['Martina', 'Giulia', 'Francesca', 'Sara']
print(uomini, quanti_uomini)  # Output: ['Luigi', 'Marco', 'Simone', 'Alessandro', 'Lorenzo'] 5

In [ ]:
from corso import verifica
verifica("E1.12")

<a id="es-e1-13"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio E1.13 · Le capitali del G8**

Il dizionario `g8` associa ai paesi del G8 la loro capitale, ma è incompleto: mancano `'Regno Unito'`
con `'Londra'` e `'Stati Uniti'` con `'Washington, D.C.'`. Vogliamo aggiungere le due coppie e poi
leggere dal dizionario il numero dei paesi, la capitale dell'Italia e l'elenco dei paesi in ordine
alfabetico. Alla fine i paesi sono otto, la capitale dell'Italia è `'Roma'` e l'elenco va da
`'Canada'` a `'Stati Uniti'`.

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Passo per passo</summary>

1. Aggiungi le coppie `'Regno Unito': 'Londra'` e `'Stati Uniti': 'Washington, D.C.'`.
2. Salva in `quanti` il numero di paesi e in `capitale_italia` la capitale dell'Italia, letta dal
   dizionario con la sua chiave.
3. Salva in `paesi` la lista dei paesi in ordine alfabetico, usando `sorted()` sulle chiavi.

</details>

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Suggerimento 1</summary>

`len()` conta le coppie di un dizionario, `g8['Italia']` legge un valore dalla sua chiave e
`sorted(g8.keys())` restituisce le chiavi in ordine alfabetico come lista.

</details>

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione E1.13**

</div>

In [ ]:
g8 = {
    'Canada': 'Ottawa',
    'Francia': 'Parigi',
    'Germania': 'Berlino',
    'Italia': 'Roma',
    'Giappone': 'Tokyo',
    'Russia': 'Mosca'
}

g8['Regno Unito'] = 'Londra'
g8['Stati Uniti'] = 'Washington, D.C.'

quanti = len(g8)
capitale_italia = g8['Italia']
paesi = sorted(g8.keys())

print(quanti, capitale_italia)  # Output: 8 Roma
print(paesi)

In [ ]:
from corso import verifica
verifica("E1.13")

<a id="es-e1-14"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio E1.14 · Sfida: una riga sul G8 (facoltativo)**

Partendo dal dizionario `g8` completo, vogliamo stampare una frase come `Il G8 ha 8 paesi e la capitale
del Giappone è Tokyo`, in cui il numero di paesi e la capitale vengono letti dal dizionario e non scritti
a mano, così la frase resta giusta se il dizionario cambia. La forma della frase è libera, quindi
l'esercizio non ha una verifica e la cella `soluzione` mostra un esempio.

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Passo per passo</summary>

1. Componi la frase con una f-string, mettendo tra graffe `len(g8)` e il valore letto con la chiave del
   paese che preferisci.
2. Cambia il paese e riesegui la cella.

</details>

<details style="margin: 14px 0;">
<summary style="cursor: pointer; padding: 4px 0;">Suggerimento 1</summary>

Dentro le graffe di una f-string può stare un'espressione qualunque, oltre al nome di una
variabile: `{len(g8)}` e `{g8['Giappone']}` funzionano.

</details>

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione E1.14**

</div>

In [ ]:
g8 = {
    'Canada': 'Ottawa',
    'Francia': 'Parigi',
    'Germania': 'Berlino',
    'Italia': 'Roma',
    'Giappone': 'Tokyo',
    'Russia': 'Mosca'
}

g8['Regno Unito'] = 'Londra'
g8['Stati Uniti'] = 'Washington, D.C.'

print(f"Il G8 ha {len(g8)} paesi e la capitale del Giappone è {g8['Giappone']}")

---

**Fine dell'esercitazione 1.** Precedente: [Ambiente, script e Ruff](../Modulo_1/Script_uv_Ruff/Ambiente_script_e_Ruff.ipynb) &nbsp;·&nbsp; Prossimo: [04 · Funzioni e controllo del flusso](04_Funzioni_e_controllo.ipynb)

<sub>Modulo 1 · Setup, Python e sintassi di base · Aula Avanzata</sub>